# Day 12: Used Car Data Preprocessing

This notebook profiles the used-car dataset, detects outliers, and builds a leakage-safe preprocessing pipeline. Numeric transformations are fitted only on the training split.

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

source_path = Path(r'c:\Users\HP\Searches\Internship\Day12_Used_Car_Preprocessing_Dataset.csv')
processed_path = source_path.with_name('Day12_Preprocessed_Used_Car_Dataset.csv')
raw = pd.read_csv(source_path)
print(f'Dataset shape: {raw.shape}')
display(raw.head())
display(raw.dtypes.to_frame('data_type'))
display(raw.isnull().sum().to_frame('missing_count'))

Dataset shape: (320, 15)


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


,data_type
Car_ID,object
Brand,object
Year,int64
Mileage_Km,int64
Engine_CC,int64
Power_BHP,float64
Fuel_Type,object
Transmission,object
City,object
Seller_Type,object


,missing_count
Car_ID,0
Brand,0
Year,0
Mileage_Km,0
Engine_CC,0
Power_BHP,0
Fuel_Type,0
Transmission,0
City,0
Seller_Type,0


## Preprocessing decisions

- `Car_ID` is an identifier, so it is excluded from model features.
- `Condition` is ordinal (`Poor` < `Fair` < `Good` < `Very Good` < `Excellent`) and is ordinal-encoded.
- Other text columns are nominal and are one-hot encoded.
- Numeric missing values use training-set medians.
- Numeric outliers are capped at training-set IQR fences instead of deleting rows, preserving the small dataset.
- Numeric features are standardized after imputation and outlier capping. The target is kept in its original lakh units.

In [14]:
numeric_columns = [
    'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners',
    'Accidents_Reported', 'Service_Score'
]
nominal_columns = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']
ordinal_columns = ['Condition']
target_column = 'Resale_Price_Lakh'
feature_columns = numeric_columns + nominal_columns + ordinal_columns

def iqr_outlier_counts(frame, columns):
    counts = {}
    for column in columns:
        first_quartile = frame[column].quantile(0.25)
        third_quartile = frame[column].quantile(0.75)
        iqr = third_quartile - first_quartile
        counts[column] = int(((frame[column] < first_quartile - 1.5 * iqr) |
                              (frame[column] > third_quartile + 1.5 * iqr)).sum())
    return pd.Series(counts).sort_values(ascending=False)

print('Duplicate rows:', raw.duplicated().sum())
print('IQR outlier counts before splitting:')
display(iqr_outlier_counts(raw, numeric_columns).to_frame('outlier_count'))
display(raw[numeric_columns + [target_column]].describe().T)

Duplicate rows: 0
IQR outlier counts before splitting:


,outlier_count
Accidents_Reported,63
Previous_Owners,14
Power_BHP,7
Engine_CC,6
Mileage_Km,2
Year,0
Service_Score,0


,count,mean,std,min,25%,50%,75%,max
Year,320.0,2019.537500,3.341367,2014.0,2017.0000,2020.00,2022.000,2025.0
Mileage_Km,320.0,74110.203125,38885.260771,700.0,46323.2500,72718.50,97951.500,320000.0
Engine_CC,320.0,1346.703125,543.408160,600.0,1004.7500,1303.00,1635.250,5000.0
Power_BHP,320.0,150.489688,36.665353,51.4,128.4500,150.75,171.475,390.0
Previous_Owners,320.0,1.668750,0.865369,1.0,1.0000,1.00,2.000,4.0
Accidents_Reported,320.0,0.243750,0.528164,0.0,0.0000,0.00,0.000,2.0
Service_Score,320.0,76.203125,12.745864,55.0,64.7500,77.00,87.000,98.0
Resale_Price_Lakh,320.0,4.963031,3.359259,1.2,2.2775,4.61,6.835,28.5


In [15]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, multiplier=1.5):
        self.multiplier = multiplier

    def fit(self, X, y=None):
        values = pd.DataFrame(X)
        first_quartile = values.quantile(0.25)
        third_quartile = values.quantile(0.75)
        iqr = third_quartile - first_quartile
        self.lower_bounds_ = (first_quartile - self.multiplier * iqr).to_numpy()
        self.upper_bounds_ = (third_quartile + self.multiplier * iqr).to_numpy()
        return self

    def transform(self, X):
        values = np.asarray(X, dtype=float)
        return np.clip(values, self.lower_bounds_, self.upper_bounds_)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array([f"x{i}" for i in range(len(self.lower_bounds_))], dtype=object)
        return np.asarray(input_features, dtype=object)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('iqr_clipper', IQRClipper()),
    ('scaler', StandardScaler()),
])
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
condition_order = [['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']]
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=condition_order, handle_unknown='use_encoded_value', unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('nominal', nominal_pipeline, nominal_columns),
    ('condition', ordinal_pipeline, ordinal_columns),
], remainder='drop', verbose_feature_names_out=False)

In [16]:
model_data = raw.drop_duplicates().copy()
X = model_data[feature_columns]
y = model_data[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit only on X_train; X_test is transformed using those learned statistics.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
processed_feature_names = preprocessor.get_feature_names_out()

print(f'Train rows: {X_train.shape[0]}, test rows: {X_test.shape[0]}')
print(f'Processed feature count: {len(processed_feature_names)}')
print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)
display(pd.DataFrame(X_train_processed, columns=processed_feature_names).head())

Train rows: 256, test rows: 64
Processed feature count: 37
Processed train shape: (256, 37)
Processed test shape: (64, 37)


,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Brand_Honda,Brand_Hyundai,Brand_Kia,...,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Condition
0,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0
1,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
2,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
3,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0
4,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0


In [17]:
processed_train = pd.DataFrame(X_train_processed, columns=processed_feature_names, index=X_train.index)
processed_test = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test.index)
processed_dataset = pd.concat([processed_train.assign(Split='train'), processed_test.assign(Split='test')])
processed_dataset[target_column] = pd.concat([y_train, y_test])
processed_dataset = processed_dataset.sort_index()
processed_dataset.to_csv(processed_path, index=False)

assert processed_dataset.isnull().sum().sum() == 0
assert processed_train.shape[1] == processed_test.shape[1]
assert set(processed_dataset['Split']) == {'train', 'test'}
assert len(processed_dataset) == len(model_data)

print('Verification passed:')
print('- Missing values:', int(processed_dataset.isnull().sum().sum()))
print('- Duplicate input rows removed:', len(raw) - len(model_data))
print('- Processed dataset shape:', processed_dataset.shape)
print(f'- Exported to: {processed_path}')
display(processed_dataset.head())

Verification passed:
- Missing values: 0
- Duplicate input rows removed: 0
- Processed dataset shape: (320, 39)
- Exported to: c:\Users\HP\Searches\Internship\Day12_Preprocessed_Used_Car_Dataset.csv


,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Brand_Honda,Brand_Hyundai,Brand_Kia,...,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Condition,Split,Resale_Price_Lakh
0,0.422486,-0.102145,-0.402934,-0.669911,-0.755752,0.0,-0.373821,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,train,6.38
1,0.119527,0.439757,-0.956625,-0.113562,-0.755752,0.0,0.830435,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,train,4.83
2,0.422486,-0.838755,0.250822,1.124864,0.484456,0.0,1.071286,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,train,7.30
3,-0.183432,-0.069952,1.636162,-0.041269,1.724664,0.0,-0.855524,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,4.0,test,3.82
4,-1.092309,0.788730,0.720014,1.756650,0.484456,0.0,0.589584,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,train,1.93
